In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset


/opt/anaconda3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# 1. SVD Compression Algorithms
# ==========================================
def truncated_svd(A, rank):
    """Return the rank-r truncated SVD approximation."""
    U, sigma, Vt = torch.linalg.svd(A, full_matrices=False)
    return U[:, :rank] @ torch.diag(sigma[:rank]) @ Vt[:rank, :]

def erc_svd(W, S, target_rank, beta=0.05):
    """ERC-SVD: redistribute rank budget to explicitly compensate the residual."""
    m, n = W.shape
    alpha = m * n / (m + n)
    residual_rank = int(beta * alpha)
    intermediate_rank = target_rank - residual_rank

    W_intermediate = truncated_svd(W @ S, rank=intermediate_rank) @ torch.linalg.inv(S)
    residual = W - W_intermediate
    residual_hat = truncated_svd(residual, rank=residual_rank)
    return W_intermediate + residual_hat

In [3]:
# ==========================================
# 2. Calibration & Replacement Logic
# ==========================================
hook_state = {
    "activation_squares": {},
    "sample_inputs": {},
    "sample_count": 0
}

def get_activation_hook(name):
    def hook(module, input, output):
        # Cast to float32 to prevent float16 overflow during squaring
        X = input[0].detach().float() 
        squared_sum = (X ** 2).sum(dim=(0, 1))
        
        if name not in hook_state["activation_squares"]:
            hook_state["activation_squares"][name] = squared_sum
            # Save the first batch for output agreement testing later
            hook_state["sample_inputs"][name] = X.cpu()
        else:
            hook_state["activation_squares"][name] += squared_sum
    return hook

def apply_erc_svd_and_refactor(module, S, target_rank):
    """Compresses the dense weight and returns a two-layer nn.Sequential."""
    W = module.weight.data.float() 
    W_erc = erc_svd(W, S, target_rank, beta=0.05) 
    
    # Factorize the resulting dense matrix into A and B
    U, sigma, Vt = torch.linalg.svd(W_erc, full_matrices=False)
    U_trunc, sigma_trunc, Vt_trunc = U[:, :target_rank], sigma[:target_rank], Vt[:target_rank, :]
    
    sqrt_sigma = torch.diag(torch.sqrt(sigma_trunc))
    B_weight = sqrt_sigma @ Vt_trunc
    A_weight = U_trunc @ sqrt_sigma
    
    layer_B = nn.Linear(module.in_features, target_rank, bias=False)
    layer_A = nn.Linear(target_rank, module.out_features, bias=module.bias is not None)
    
    layer_B.weight.data = B_weight.to(module.weight.dtype)
    layer_A.weight.data = A_weight.to(module.weight.dtype)
    if module.bias is not None:
        layer_A.bias.data = module.bias.data
        
    return nn.Sequential(layer_B, layer_A)

In [4]:
# ==========================================
# 3. Main Execution Pipeline
# ==========================================
def main():
    model_path = "/Users/sathya/Downloads/Llama2-7b-hf" # UPDATE THIS
    target_rank = 1024
    
    print("1. Loading Model and Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path, 
        device_map="cpu", 
        torch_dtype=torch.float16,
        local_files_only=True
    )

    print("2. Preparing WikiText Calibration Dataset...")
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    # Filter for substantial sentences and take 16 samples
    texts = [text for text in dataset["text"] if len(text.strip()) > 50][:16]
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
    hook_state["sample_count"] = inputs["input_ids"].numel()

    print("3. Attaching Hooks and Running Forward Pass...")
    hooks = []
    # Targeting the first 2 layers for demonstration
    for i in range(2):
        name = f"layer_{i}_down_proj"
        module = model.model.layers[i].mlp.down_proj
        hooks.append(module.register_forward_hook(get_activation_hook(name)))

    with torch.no_grad():
        model(**inputs)

    for h in hooks:
        h.remove()

    print("\n4. Applying ERC-SVD and Computing Metrics...")
    for i in range(2):
        name = f"layer_{i}_down_proj"
        original_module = model.model.layers[i].mlp.down_proj
        
        # Compute S (add epsilon to ensure invertibility)
        sq_sum = hook_state["activation_squares"][name]
        mean_sq = sq_sum / hook_state["sample_count"]
        S = torch.diag(torch.sqrt(mean_sq) + 1e-6).to(original_module.weight.device)
        
        # Compress
        new_module = apply_erc_svd_and_refactor(original_module, S, target_rank).to(model.device)
        
        # Evaluate Agreement
        X_test = hook_state["sample_inputs"][name].to(model.device).to(original_module.weight.dtype)
        with torch.no_grad():
            Y_orig = original_module(X_test)
            Y_comp = new_module(X_test)
            
        rel_error = torch.linalg.norm(Y_orig - Y_comp) / torch.linalg.norm(Y_orig)
        cos_sim = F.cosine_similarity(Y_orig, Y_comp, dim=-1).mean()
        
        # Evaluate Compression
        in_f, out_f = original_module.in_features, original_module.out_features
        orig_p = in_f * out_f
        comp_p = target_rank * (in_f + out_f)
        
        print(f"\n--- {name} ---")
        print(f"Parameters:        {orig_p:,} -> {comp_p:,}")
        print(f"Compression Ratio: {orig_p / comp_p:.2f}x smaller")
        print(f"Relative Error:    {rel_error.item():.4f}")
        print(f"Cosine Similarity: {cos_sim.item():.4f}")

        # Inject into model
        model.model.layers[i].mlp.down_proj = new_module

In [5]:
if __name__ == "__main__":
    main()

1. Loading Model and Tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 35038.83it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2. Preparing WikiText Calibration Dataset...


3. Attaching Hooks and Running Forward Pass...

4. Applying ERC-SVD and Computing Metrics...

--- layer_0_down_proj ---
Parameters:        45,088,768 -> 15,466,496
Compression Ratio: 2.92x smaller
Relative Error:    0.1495
Cosine Similarity: 0.9702

--- layer_1_down_proj ---
Parameters:        45,088,768 -> 15,466,496
Compression Ratio: 2.92x smaller
Relative Error:    0.0035
Cosine Similarity: 0.9194


In [7]:
def apply_truncated_svd_and_refactor(module, target_rank):
    """Compresses the dense weight using standard SVD and returns an nn.Sequential."""

    W = module.weight.data.float() 
    
    # 1. Standard Truncated SVD (using your original function)
    W_trunc = truncated_svd(W, target_rank)
    
    # 2. Factorize the dense matrix into A and B
    U, sigma, Vt = torch.linalg.svd(W_trunc, full_matrices=False)
    U_trunc, sigma_trunc, Vt_trunc = U[:, :target_rank], sigma[:target_rank], Vt[:target_rank, :]
    
    sqrt_sigma = torch.diag(torch.sqrt(sigma_trunc))
    B_weight = sqrt_sigma @ Vt_trunc
    A_weight = U_trunc @ sqrt_sigma
    
    # 3. Create the replacement layers
    layer_B = nn.Linear(module.in_features, target_rank, bias=False)
    layer_A = nn.Linear(target_rank, module.out_features, bias=module.bias is not None)
    
    layer_B.weight.data = B_weight.to(module.weight.dtype)
    layer_A.weight.data = A_weight.to(module.weight.dtype)
    if module.bias is not None:
        layer_A.bias.data = module.bias.data
        
    return nn.Sequential(layer_B, layer_A)

# ==========================================
# Execution Loop for Jupyter Cell
# ==========================================
# (Optional) Reload model here if you already modified it in a previous cell:
# model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16, local_files_only=True)

target_rank = 1024
model_path = "/Users/sathya/Downloads/Llama2-7b-hf" # UPDATE THIS
target_rank = 1024

print("1. Loading Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map="cpu", 
    torch_dtype=torch.float16,
    local_files_only=True
)
print("Applying Standard Truncated SVD and Computing Metrics...\n")
for i in range(2):
    name = f"layer_{i}_down_proj"
    original_module = model.model.layers[i].mlp.down_proj
    
    # Compress
    new_module_trunc = apply_truncated_svd_and_refactor(original_module, target_rank).to(model.device)
    
    # Evaluate Agreement (reusing X_test from the previous calibration cell)
    X_test = hook_state["sample_inputs"][name].to(model.device).to(original_module.weight.dtype)
    with torch.no_grad():
        Y_orig = original_module(X_test)
        Y_comp = new_module_trunc(X_test)
        
    rel_error = torch.linalg.norm(Y_orig - Y_comp) / torch.linalg.norm(Y_orig)
    cos_sim = F.cosine_similarity(Y_orig, Y_comp, dim=-1).mean()
    
    # Evaluate Compression
    in_f, out_f = original_module.in_features, original_module.out_features
    orig_p = in_f * out_f
    comp_p = target_rank * (in_f + out_f)
    
    print(f"--- {name} (Standard SVD) ---")
    print(f"Parameters:        {orig_p:,} -> {comp_p:,}")
    print(f"Compression Ratio: {orig_p / comp_p:.2f}x smaller")
    print(f"Relative Error:    {rel_error.item():.4f}")
    print(f"Cosine Similarity: {cos_sim.item():.4f}\n")

    # Swap the module in the model
    model.model.layers[i].mlp.down_proj = new_module_trunc

1. Loading Model and Tokenizer...


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 61600.00it/s]

Applying Standard Truncated SVD and Computing Metrics...



--- layer_0_down_proj (Standard SVD) ---
Parameters:        45,088,768 -> 15,466,496
Compression Ratio: 2.92x smaller
Relative Error:    0.5117
Cosine Similarity: 0.8794

--- layer_1_down_proj (Standard SVD) ---
Parameters:        45,088,768 -> 15,466,496
Compression Ratio: 2.92x smaller
Relative Error:    0.2130
Cosine Similarity: 0.7603

